In [82]:
from dotenv import load_dotenv
from openai import OpenAI
import os
import json
from pypdf import PdfReader
import gradio as gr

In [83]:
load_dotenv(override=True)

True

In [84]:
## Prep for system prompt - Part 1
reader = PdfReader("resources/Linkedin_Summary.pdf")
linkedin_summary = ""
for page in reader.pages:
    text = page.extract_text()
    if(text):
        linkedin_summary += text

profile_summary = ""

with open("resources/profile_summary.txt", "r", encoding = "utf-8") as f:
    profile_summary = f.read()



In [85]:
## Prep for system prompt - Part 2
system_prompt = f"""
You're a career digital twin of Madhan. You will act as Madhan (in first person). Your job is to respond professionaly to people on behalf of Madhan when they ask about Madhan's work related things.
Do not answer any other kind of questions. Here's a summary of Madhan's Linkedin summary and overall profile summary. Always stick to this while you're answering.
Do not hallucinate and say anything that's not in the summary provided below. For an unknown question, use the tools appropriately. 

Madhan's Linkedin Summary:

{linkedin_summary}

Madhan's overall profile Summary:

{profile_summary}

SOURCE OF TRUTH

When answering questions about experience, always calculate from employment dates instead of using the textual summaries.

Relevant product experience:

- Product Manager, BYJU'S: June 2022 - August 2023
- Product Owner, Accenture: March 2021 - May 2022
- AI Product Manager, Aera: April 2025 - Present

Do not use "7+ years" or "8+ years" statements when answering questions about experience.

If a question asks about years of experience, calculate it from these dates.

Important Guardrails for chat:

1. Always stick to the summaries provided above. Do not hallucinate. If you don't know something, say you don't know.
2. Always be professional, and do not envourage any non-career related information of Madhan. 
3. Do not answer any other kind of questions.
4. Act as Madhan (in first person). You're the digital twin of Madhan.
5. Always be polite.
6. Keep the conversation human-like. 
7. Do not use any unprofessional language or words. 
8. Always think, reason, before answering user's question.

"""

system = [{"role":"system", "content": system_prompt}]

In [86]:
print(system_prompt)


You're a career digital twin of Madhan. You will act as Madhan (in first person). Your job is to respond professionaly to people on behalf of Madhan when they ask about Madhan's work related things.
Do not answer any other kind of questions. Here's a summary of Madhan's Linkedin summary and overall profile summary. Always stick to this while you're answering.
Do not hallucinate and say anything that's not in the summary provided below. For an unknown question, use the tools appropriately. 

Madhan's Linkedin Summary:

   
Contact
madhan.e4@gmail.com
www.linkedin.com/in/madhan-
narayanaswami (LinkedIn)
Top Skills
Global Client Management
Requirements Analysis
Product roadmapping
Languages
Tamil
English
Hindi
Certifications
Data Analysis with Python
Product Strategy
User Experience Research and
Design Specialization
Data Visualization with Tableau
Specialization 
Data Science Professional
Certificate
Madhan Narayanaswami
Product Manager - AI Initiatives and Platform Experience @ Aera |


In [87]:
## Define tools and manager

def push_unknown_question(question):
    with open("resources/unknown_questions.txt", "a", encoding = "utf-8") as f:
        f.write()

def push_user_details(details):
    with open("resources/record_user_details.txt", "a", encoding = "utf-8") as f:
        f.write(details)


def record_unknown_question(question):
    push_unknown_question(question)
    return "Recorded question"

def record_user_details(email, notes = "Not provided"):
    push_user_details(f"User's email is {email} and notes is {notes}")

record_unknown_question_tool = {
    "name": "record_unknown_question",
    "description": "This tool can be used to record an unknown question",
    "parameters":{
        "type": "object",
        "properties":{
            "question":{"type":"string", "description":"The question that couldn't be answered."}
        },
        "required":["question"],
        "additionalProperties":False
    }
}

record_record_user_details_tool = {
    "name": "record_user_details",
    "description": "This tool can be used to record details of users who want to reach out",
    "parameters":{
        "type": "object",
        "properties":{
            "email":{"type":"string", "description":"Email ID of the user who wants to reach out."},
            "notes":{"type":"string", "description":"Any extra notes that the user might want to provide."}
        },
        "required":["email"],
        "additionalProperties":False
    }
}

tools = [{"type":"function", "function":record_unknown_question_tool}, {"type":"function", "function":record_record_user_details_tool}]

tool_map = {"record_unknown_question":record_unknown_question, "record_user_details":record_user_details}

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        result = tool_map.get(function_name)(**arguments)
        results.append(
            {"role":"tool", "content":json.dumps(result), "tool_call_id":tool_call.id}
        )
    return results


In [89]:
## Defining LLM, Chat and main module

openai = OpenAI()

def chat(message, history):
    messages = system + history + [{"role":"user", "content": message}]
    response = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = messages,
        tools = tools
    )
    while response.choices[0].finish_reason == "tool_calls":
        tool_calls = response.choices[0].message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(response.choices[0].message)
        messages.extend(results)
        response = openai.chat.completions.create(
            model = "gpt-4o-mini",
            messages = messages,
            tools = tools
        )
    return response.choices[0].message.content


gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/opt/anaconda3/envs/Feb2/lib/python3.14/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/opt/anaconda3/envs/Feb2/lib/python3.14/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
  File "/opt/anaconda3/envs/Feb2/lib/python3.14/site-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/opt/anaconda3/envs/Feb2/lib/python3.14/site-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
        pool_request.request
    )
  File "/opt/anaconda3/envs/Feb2/lib/python3.14/site-packages/httpcore/_sync/connection.py", line 101, in handle_request
    raise exc
  File "/opt/anaconda3/envs/Feb2/lib/python3.14/site-packages/httpcore/_sync/connection.py", line 78, in handle_request
    stream = self._connect(request)
  File "/opt/anaco